In [1]:
import re
import sys
import json
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
pd.options.mode.chained_assignment = None  # default='warn'

from us import states
from census import Census

from setup import CENSUS_API_KEY

In [2]:
target_states = {
    "Connecticut": str(states.CT.fips),
    "Maine": str(states.ME.fips),
    "Massachusetts": str(states.MA.fips),
    "New Hampshire": str(states.NH.fips),
    "Rhode Island": str(states.RI.fips),
    "Vermont": str(states.VT.fips)
}

target_municipalities = [
    # Connecticut
    "Groton town, New London County, Connecticut",
    "New Haven town, New Haven County, Connecticut",
    "Norwalk town, Fairfield County, Connecticut",

    # Massachusetts
    "Amherst town, Hampshire County, Massachusetts",
    "Arlington town, Middlesex County, Massachusetts",
    "Beverly city, Essex County, Massachusetts",
    "Boston city, Suffolk County, Massachusetts",
    "Cambridge city, Middlesex County, Massachusetts",
    "Concord town, Middlesex County, Massachusetts",
    "Dedham town, Norfolk County, Massachusetts",
    "Greenfield city, Franklin County, Massachusetts",
    "Lexington town, Middlesex County, Massachusetts",
    "Medford city, Middlesex County, Massachusetts",
    "New Bedford city, Bristol County, Massachusetts",
    "Northampton city, Hampshire County, Massachusetts",
    "Provincetown town, Barnstable County, Massachusetts",
    "Somerville city, Middlesex County, Massachusetts",
    "Winchester town, Middlesex County, Massachusetts",

    # Maine
    "Acton town, York County, Maine",
    "Alfred town, York County, Maine",
    "Arundel town, York County, Maine",
    "Baldwin town, Cumberland County, Maine",
    "Bath city, Sagadahoc County, Maine",
    "Berwick town, York County, Maine",
    "Biddeford city, York County, Maine",
    "Brownfield town, Oxford County, Maine",
    "Buxton town, York County, Maine",
    "Cornish town, York County, Maine",
    "Dayton town, York County, Maine",
    "Denmark town, Oxford County, Maine",
    "Eliot town, York County, Maine",
    "Fryeburg town, Oxford County, Maine",
    "Hiram town, Oxford County, Maine",
    "Hollis town, York County, Maine",
    "Kennebunk town, York County, Maine",
    "Kennebunkport town, York County, Maine",
    "Kittery town, York County, Maine",
    "Lebanon town, York County, Maine",
    "Limerick town, York County, Maine",
    "Limington town, York County, Maine",
    "Lovell town, Oxford County, Maine",
    "Lyman town, York County, Maine",
    "Newfield town, York County, Maine",
    "North Berwick town, York County, Maine",
    "Ogunquit town, York County, Maine",
    "Old Orchard Beach town, York County, Maine",
    "Parsonsfield town, York County, Maine",
    "Porter town, Oxford County, Maine",
    "Portland city, Cumberland County, Maine",
    "Saco city, York County, Maine",
    "Sanford city, York County, Maine",
    "Shapleigh town, York County, Maine",
    "South Berwick town, York County, Maine",
    "South Portland city, Cumberland County, Maine",
    "Stoneham town, Oxford County, Maine",
    "Stow town, Oxford County, Maine",
    "Sweden town, Oxford County, Maine",
    "Waterboro town, York County, Maine",
    "Wells town, York County, Maine",
    "York town, York County, Maine",

    # New Hampshire
    "Dover city, Strafford County, New Hampshire",
    "Exeter town, Rockingham County, New Hampshire",
    "Hanover town, Grafton County, New Hampshire",
    "Keene city, Cheshire County, New Hampshire",
    "Lebanon city, Grafton County, New Hampshire",
    "Nashua city, Hillsborough County, New Hampshire",
    "Portsmouth city, Rockingham County, New Hampshire",

    # Rhode Island
    "Cranston city, Providence County, Rhode Island",
    "Pawtucket city, Providence County, Rhode Island",
    "Providence city, Providence County, Rhode Island",

    # Vermont
    "Bolton town, Chittenden County, Vermont",
    "Brattleboro town, Windham County, Vermont",
    "Buels gore, Chittenden County, Vermont",
    "Burlington city, Chittenden County, Vermont",
    "Charlotte town, Chittenden County, Vermont",
    "Colchester town, Chittenden County, Vermont",
    "Essex town, Chittenden County, Vermont",
    "Essex Junction city, Chittenden County, Vermont", 
    "Hartford town, Windsor County, Vermont",
    "Hinesburg town, Chittenden County, Vermont",
    "Huntington town, Chittenden County, Vermont",
    "Jericho town, Chittenden County, Vermont",
    "Milton town, Chittenden County, Vermont",
    "Richmond town, Chittenden County, Vermont",
    "Shelburne town, Chittenden County, Vermont",
    "South Burlington city, Chittenden County, Vermont",
    "St. George town, Chittenden County, Vermont",
    "Underhill town, Chittenden County, Vermont",
    "Westford town, Chittenden County, Vermont",
    "Williston town, Chittenden County, Vermont",
    "Winooski city, Chittenden County, Vermont"
]

def check_nems_membership(name: str):
    if name in target_municipalities:
        return True
    return False

target_years = [
    2024
]

In [3]:
def parse_acs5_cousub_name(name_string):

        # Split on commas first
        parts = name_string.split(",")

        # Left side:
        # "Exeter town"
        left = parts[0].strip()

        # County:
        # "Rockingham County"
        county = parts[1].replace("County", "").strip()

        # State:
        # "New Hampshire"
        state = parts[2].strip()

        # Split municipality name/type
        left_parts = left.split()

        muni_type = left_parts[-1]

        name = " ".join(left_parts[:-1])

        return pd.Series([
            name,
            muni_type,
            county,
            state
        ])

def acs_df_from_raw(all_raw_data, rename_dict:dict):
    output = pd.DataFrame(all_raw_data)
    output["GEOID"] = output["state"] + output["county"] + output["county subdivision"]

    output["nems"] = output["NAME"].apply(check_nems_membership)
    output.rename(columns=rename_dict, inplace=True)


    output[
            ["name_str", "muni_str", "county_str", "state_str"]
        ] = output["NAME"].apply(parse_acs5_cousub_name)
    output.drop(columns=["NAME"], inplace=True)
    output.sort_values(by=["GEOID", "year"], inplace=True)

    return output

In [30]:
# metadata: https://api.census.gov/data/2024/acs/acs5/groups/B25036.json
# helper functions for age-related variables
def get_b25036_metadata(year):
    """
    Download universal ACS metadata and filter for B25036 variables.

    This endpoint is stable across ACS vintages.
    """

    url = f"https://api.census.gov/data/{year}/acs/acs5/groups/B25036.json"+"?key="+CENSUS_API_KEY

    response = requests.get(url)
    response.raise_for_status()

    all_variables = response.json().get("variables", {})

    b25036_metadata = {
        variable: info
        for variable, info in all_variables.items()
        if (
            variable.startswith("B25036_") and variable.endswith("E") 
            )
    }

    return b25036_metadata


def parse_construction_range(label):

    if not isinstance(label, str):
        return None

    normalized = (
        label.lower()
        .replace("-", " ")
        .replace("—", " ")
        .replace("–", " ")
    )

    built_match = re.search(r"built (.*)", normalized)

    if built_match is None:
        return None

    built_text = built_match.group(1)

    # Built aaaa to bbbb
    match = re.search(r"(\d{4}) to (\d{4})", built_text)
    if match:
        return int(match.group(1)), int(match.group(2))

    # Built xxxx or later
    match = re.search(r"(\d{4}) or later", built_text)
    if match:
        return int(match.group(1)), None

    # Built yyyy or earlier
    match = re.search(r"(\d{4}) or earlier", built_text)
    if match:
        return None, int(match.group(1))

    return None


def get_relevant_variables(year):
    """
    Construct a metadata dictionary for renter-occupied construction periods.
    """

    # universe, 
    metadata = get_b25036_metadata(year)

    parsed_variables = {}

    for variable, info in metadata.items():

        label = info.get("label", "")
        # print("\n", label, "\n")

        # Restrict to renter-occupied housing units
        if "Renter occupied" not in label:
            continue

        parsed_range = parse_construction_range(label)
        # print("\n", parsed_range, "\n")

        if parsed_range is None:
            continue

        start_year, end_year = parsed_range

        def rename_var(start_year, end_year):
            if start_year is None:
                return "built_pre_" + str(end_year)
            
            if end_year is None:
                return "built_" + str(start_year) + "_to_" + str(year)
            
            return "built_" + str(start_year) + "_to_" + str(end_year)
        
        parsed_variables[variable] = rename_var(start_year, end_year)
        # {
        #     "start_year": start_year,
        #     "end_year": end_year,
        #     "label": label,
        #     "to_be_called": rename_var(start_year, end_year)
        # }

    return parsed_variables #universe, parsed_variables


def get_housing_age_data(
        target_states, target_years, loud=False
        ):
    c = Census(CENSUS_API_KEY)

    all_raw_data = []
    # df = pd.DataFrame()

    if loud:
        print("\t\tquerying: HOUSING AGE")
    
    for year in target_years:

        try:

            if loud:
                print(f"\n -- processing metadata for {year}.")
            
            # new_universe, 
            age_variables = get_relevant_variables(year) |{"B25003_003E": "rent_occ"}
            # all_universes.add(new_universe)
            
            fields = ["NAME"] + list(age_variables.keys())
            
            if loud:
                print(
                    "\tdone. metadata is " +
                    ("not empty." if bool(age_variables) else "empty.")
                    )

            # Loop through each state
            for state_name, state_fips in target_states.items():

                if loud:
                    print(f" -- grabbing {state_name}.")
                
                raw_data = c.acs5.state_county_subdivision(
                    fields,
                    state_fips,
                    Census.ALL,
                    Census.ALL,
                    year=year
                )
                
                for row in raw_data:
                    row["year"] = year

                all_raw_data.extend(raw_data)

        except Exception as e:

            print(f"    [!] Error completely skipping year {year}: {e}")

    output = acs_df_from_raw(all_raw_data, age_variables)
    """ 
    WARNING: 
        Very hacky name changing here.
        Would love to find a better approach, but this is what I've got for now.
    """
    output.loc[output['name_str']=='Amherst Town', 'nems']=True
    output.loc[output['name_str']=='Amherst Town', 'name_str']='Amherst'

    # "Groton town, New London County, Connecticut",
    # "New Haven town, New Haven County, Connecticut",
    # "Norwalk town, Fairfield County, Connecticut"
    output.loc[(output['name_str']=='Groton') & (output['state_str']=="Connecticut"), 'nems'] = True
    output.loc[(output['name_str']=='New Haven') & (output['state_str']=="Connecticut"), 'nems'] = True
    output.loc[(output['name_str']=='Norwalk') & (output['state_str']=="Connecticut"), 'nems'] = True

    output.sort_values(by=["GEOID", "year"], inplace=True)
    first_cols = ["year", "GEOID", "name_str"]
    last_cols = [col for col in output.columns if col not in first_cols]
    output = output[first_cols + last_cols]

    return output    



def query_states_years_variables(target_states, target_years, variables, 
        loud=True):
    
    c = Census(CENSUS_API_KEY)
    fields = ["NAME"] + list(variables.keys())

    all_raw_data = []

    if loud:
        print("\n\t\tquerying:\n")

    for year in target_years:

        for state_name, state_fips in target_states.items():
            if loud:
                print(f" -- grabbing {state_name} for {year}.")
            
            try:
                # Query the API
                raw_data = c.acs5.state_county_subdivision(
                    fields, 
                    state_fips, 
                    Census.ALL, 
                    Census.ALL, 
                    year=year
                )
                
                for row in raw_data:
                    row["year"] = year
                
                all_raw_data.extend(raw_data)
                
            except Exception as e:
                print(f"    [!] Error fetching {state_name} in {year}: {e}")


    output = acs_df_from_raw(all_raw_data, variables)
    # output.drop(output[output["county subdivision"] == "00000"].index, inplace=True)
    """ 
    WARNING: 
        Very hacky name changing here.
        Would love to find a better approach, but this is what I've got for now.
    """
    output.loc[output['name_str']=='Amherst Town', 'nems']=True
    output.loc[output['name_str']=='Amherst Town', 'name_str']='Amherst'

    # "Groton town, New London County, Connecticut",
    # "New Haven town, New Haven County, Connecticut",
    # "Norwalk town, Fairfield County, Connecticut"
    output.loc[(output['name_str']=='Groton') & (output['state_str']=="Connecticut"), 'nems'] = True
    output.loc[(output['name_str']=='New Haven') & (output['state_str']=="Connecticut"), 'nems'] = True
    output.loc[(output['name_str']=='Norwalk') & (output['state_str']=="Connecticut"), 'nems'] = True
    
    return output




def get_heating_data():
    print("Caleb did this.")
    pass

def get_landscape_data(target_states, target_years, variables, loud=True):


    local_variables = {
    "B01003_001E": "total_population",

    "B25003_001E": "tot_occ",
    "B25003_003E": "rent_occ"
    }
    
    output = query_states_years_variables(
        target_states, target_years, variables, 
        loud=loud)
    
    output["rent_sh"] = round(100.00*output["rent_occ"]/output["tot_occ"], 2)


    output["rent_lt_30"] = 100.00*(
        output["perc_rent_of_income_0-10"] + \
        output["perc_rent_of_income_10-14.9"] + \
        output["perc_rent_of_income_15-19.9"] + \
        output["perc_rent_of_income_20-24.9"] + \
        output["perc_rent_of_income_25-29.9"]
    )/output["rent_occ"]

    output["rent_gt_30"] = 100.00*(
        output["perc_rent_of_income_30-34.9"] + \
        output["perc_rent_of_income_35-39.9"] + \
        output["perc_rent_of_income_40-44.9"] + \
        output["perc_rent_of_income_50-100"]
    )/output["rent_occ"]

    output.drop(columns=[
        "perc_rent_of_income_0-10",
        "perc_rent_of_income_10-14.9",
        "perc_rent_of_income_15-19.9",
        "perc_rent_of_income_20-24.9",
        "perc_rent_of_income_25-29.9",
        "perc_rent_of_income_30-34.9",
        "perc_rent_of_income_35-39.9",
        "perc_rent_of_income_40-44.9",
        "perc_rent_of_income_50-100"
    ],inplace=True)

    output.sort_values(by=["GEOID", "year"], inplace=True)
    first_cols = [
        "year", "GEOID", "tot_occ", "rent_occ", "rent_sh", 
        "rent_gt_30", "rent_lt_30"]
    last_cols = [col for col in output.columns if col not in first_cols]
    output = output[first_cols + last_cols]

    return output



# combine age- and non-age-related data into one df
def get_all_data(target_states, target_municipalities, target_years, variables):
    old_stdout = sys.stdout
    log_file = open("pipeline_nb.log", "w")     # NOTE: different name for notebook version
    sys.stdout = log_file

    df_age = get_housing_age_data(target_states, target_municipalities, target_years)
    df_age = df_age[
        ['year', 'GEOID', 'built_2020_to_2024', 'built_2010_to_2019',
       'built_2000_to_2009', 'built_1990_to_1999', 'built_1980_to_1989',
       'built_1970_to_1979', 'built_1940_to_1949', 'built_pre_1939',
       'built_1960_to_1969', 'built_1950_to_1959']
    ]

    df_heat = None #get_heating_data(.......)

    df_non_age = get_non_age_data(target_states, target_municipalities, target_years, variables, loud=True)
    output = df_age.merge(
        df_non_age, 
        how='inner',
        on=["year", "GEOID"]
        )
    
    sys.stdout = old_stdout
    log_file.close()

    return output.sort_values(by=["GEOID", "year"])

In [5]:
# variables: https://api.census.gov/data/2024/acs/acs5/variables.html
landscape_variables = {
    "B01003_001E": "total_population",

    "B25003_001E": "tot_occ",
    "B25003_003E": "rent_occ",

    # income
    "B25119_003E": "med_renter_income",
    
    "B25070_002E": "perc_rent_of_income_0-10",
    "B25070_003E": "perc_rent_of_income_10-14.9",
    "B25070_004E": "perc_rent_of_income_15-19.9",
    "B25070_005E": "perc_rent_of_income_20-24.9",
    "B25070_006E": "perc_rent_of_income_25-29.9",
    "B25070_007E": "perc_rent_of_income_30-34.9",
    "B25070_008E": "perc_rent_of_income_35-39.9",
    "B25070_009E": "perc_rent_of_income_40-44.9",
    "B25070_010E": "perc_rent_of_income_50-100",
    "B25070_011E": "perc_rent_of_income_not_computed"
}

heating_variables = {
    # heating fuel
    # NOTE: not renter specific
    "B25040_002E": "heating_utility_gas",
    "B25040_003E": "heating_bottled_gas",
    "B25040_004E": "heating_electricity",
    "B25040_005E": "heating_oil",
    "B25040_006E": "heating_coal_coke",
    "B25040_007E": "heating_wood",
    "B25040_008E": "heating_solar",
    "B25040_009E": "heating_other",
    "B25040_010E": "heating_none"
}


In [52]:

housing_size_variables = {
    "B25003_003E": "rent_occ",

    # rental housing type
    "B25032_014E": "detached_units_1",
    "B25032_015E": "attached_units_1",
    "B25032_016E": "attached_units_2",
    "B25032_017E": "attached_units_3-4",

    "B25032_018E": "attached_units_5-9",
    "B25032_019E": "attached_units_10-19",
    "B25032_020E": "attached_units_20-49",
    "B25032_021E": "attached_units_50+"
}

def get_housing_size_data(target_states, target_years, housing_size_variables, loud=True):
    
    output = query_states_years_variables(target_states, target_years, housing_size_variables, loud=False)
    output["units_1"] = 100.00* (output["detached_units_1"] + output["attached_units_1"]) /output["rent_occ"]
    output["units_2-4"] = 100.00* (output["attached_units_2"] + output["attached_units_3-4"]) /output["rent_occ"]

    output["units_5+"] = 100.00* (
        output["attached_units_5-9"] + \
        output["attached_units_10-19"] + \
        output["attached_units_20-49"] + \
        output["attached_units_50+"]
        )/output["rent_occ"]
    
    output = output.drop(columns=[
        "detached_units_1",
        "attached_units_1",
        "attached_units_2",
        "attached_units_3-4",
        "attached_units_5-9", 
        "attached_units_10-19", 
        "attached_units_20-49", 
        "attached_units_50+"
        ])

    return output

size_df = get_housing_size_data(
    target_states=target_states,
    target_years=target_years,
    housing_size_variables=housing_size_variables,
    loud=False)
size_df = size_df[["GEOID", "year", "units_1", "units_2-4", "units_5+", "nems"]]
size_df["most_size"] = size_df[["units_1", "units_2-4", "units_5+"]].idxmax(axis='columns')

age_df = get_housing_age_data(
    target_states=target_states,
    target_years=target_years,
    loud=False)

age_df["modern"] = 100.00* (age_df["built_2010_to_2019"] + age_df["built_2020_to_2024"]) / age_df["rent_occ"]
age_df["mid"] = 100.00* (age_df["built_2000_to_2009"] + age_df["built_1990_to_1999"] + age_df["built_1980_to_1989"]) / age_df["rent_occ"]
age_df["legacy"] = 100.00* (age_df["built_1970_to_1979"] + age_df["built_1960_to_1969"] + age_df["built_1950_to_1959"] +\
    age_df["built_1940_to_1949"] + age_df["built_pre_1939"]) / age_df["rent_occ"]

age_df["most_age"] = age_df[["modern", "mid", "legacy"]].idxmax(axis='columns')

age_df.drop(columns=[
    'built_2020_to_2024', 'built_2010_to_2019',
       'built_2000_to_2009', 'built_1990_to_1999', 'built_1980_to_1989',
       'built_1970_to_1979', 'built_1940_to_1949', 'built_pre_1939',
       'built_1960_to_1969', 'built_1950_to_1959'
], inplace=True)

age_size_df = age_df.merge(size_df, on=["year", "GEOID"], how="inner")
age_size_df["nems"] = age_size_df["nems_x"]
age_size_df.drop(columns=["nems_x","nems_y"],inplace=True)


/var/folders/tl/gk43p8c13s9_wstnvb4prcf80000gn/T/ipykernel_99702/3534909788.py:48: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  size_df["most_size"] = size_df[["units_1", "units_2-4", "units_5+"]].idxmax(axis='columns')
/var/folders/tl/gk43p8c13s9_wstnvb4prcf80000gn/T/ipykernel_99702/3534909788.py:60: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  age_df["most_age"] = age_df[["modern", "mid", "legacy"]].idxmax(axis='columns')


In [54]:
# Source: https://www.census.gov/cgi-bin/geo/shapefiles/index.php

gdf = gpd.GeoDataFrame(
    pd.concat([
        gpd.read_file('./shapes/tl_2024_09_cousub'),
        gpd.read_file('./shapes/tl_2024_23_cousub'), 
        gpd.read_file('./shapes/tl_2024_25_cousub'),
        gpd.read_file('./shapes/tl_2024_33_cousub'),
        gpd.read_file('./shapes/tl_2024_44_cousub'),
        gpd.read_file('./shapes/tl_2024_50_cousub'),
    ])
)

age_size_gdf = gdf.merge(
    age_size_df, 
    on="GEOID", 
    how="inner"
    )

ct_state = gpd.read_file(states.CT.shapefile_urls()['state'])
me_state = gpd.read_file(states.ME.shapefile_urls()['state'])
ma_state = gpd.read_file(states.MA.shapefile_urls()['state'])
nh_state = gpd.read_file(states.NH.shapefile_urls()['state'])
ri_state = gpd.read_file(states.RI.shapefile_urls()['state'])
vt_state = gpd.read_file(states.VT.shapefile_urls()['state'])

In [ ]:
nems_gdf = gdf.merge(
    age_size_df.loc[ age_size_df["nems"]==True ], on="GEOID", how="inner"
    )

non_nems_gdf = gdf.merge(
    age_size_df.loc[ age_size_df["nems"]==False ], on="GEOID", how="inner"
    )

fig, ax = plt.subplots(figsize=(10,10))
ax.axis('off')

ct_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)
me_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)
ma_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)
nh_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)
ri_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)
vt_state.plot(ax=ax, color='white', edgecolor='k', linewidth=2)

gdf.plot(ax=ax, color='white', edgecolor='k', linewidth=0.5)
nems_gdf.plot(ax=ax, column="legacy", cmap='Blues', legend=True, edgecolor=None)
non_nems_gdf.plot(ax=ax, column="legacy", cmap='Greens', legend=True, edgecolor=None)